# Person 3: Thai Character Classification with MobileNetV3-Large

Notebook สำหรับ Google Colab T4 โดยใช้ Transfer Learning กับ MobileNetV3-Large

แนวทางหลัก: ใช้ split แบบ 80/20 ที่ทีม Data เตรียมไว้, ใช้ภาพ augmented เฉพาะฝั่ง Train, รักษาอัตราส่วนภาพด้วย letterbox, ใช้ class-weighted loss + label smoothing, ฝึกแบบสองช่วง และหยุดก่อนกำหนดเมื่อ Validation ไม่ดีขึ้น

> สำคัญ: โฟลเดอร์ `split_80_20/test` ใน ZIP ทำหน้าที่เป็น **Validation 20%** สำหรับงานนี้ ส่วน Test 5% ของอาจารย์เป็นชุดแยกและห้ามใช้เลือกโมเดล

## 1. ตั้งค่า Colab

เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรัน แล้ววาง `TH_Characters.zip` ไว้ใน Google Drive ตามค่า `ZIP_PATH` ด้านล่าง

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi


In [ ]:
from pathlib import Path

# แก้เฉพาะบรรทัดนี้ ถ้าเก็บ ZIP ไว้ในโฟลเดอร์อื่นของ Google Drive
ZIP_PATH = Path('/content/drive/MyDrive/dl/TH_Characters.zip')
LOCAL_ZIP = Path('/content/TH_Characters.zip')
EXTRACT_ROOT = Path('/content/thai_char_data')
SPLIT_ROOT = EXTRACT_ROOT / 'TH_Characters/Train_Test-Base/split_80_20'
TRAIN_DIR = SPLIT_ROOT / 'train_aug'
VAL_DIR = SPLIT_ROOT / 'test'  # Validation 20% ของงานนี้
LABELS_CSV = EXTRACT_ROOT / 'TH_Characters/Train_Test-Base/labels.csv'
SUMMARY_CSV = SPLIT_ROOT / 'summary_aug.csv'
OUTPUT_DIR = Path('/content/drive/MyDrive/dl/thai_char_person3_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f'ไม่พบไฟล์: {ZIP_PATH}'
print('ZIP:', ZIP_PATH)
print('Outputs:', OUTPUT_DIR)


## 2. คัดลอก ZIP และแตกเฉพาะข้อมูลที่ใช้

ZIP มีไฟล์มากกว่า 5 ล้านไฟล์ จึงไม่แตกทั้งหมด Notebook จะดึงเฉพาะ `train_aug`, validation และไฟล์สรุป การทำครั้งแรกอาจใช้เวลาหลายนาที แต่รอบถัดไปจะข้ามขั้นตอนนี้หากข้อมูลยังอยู่ใน Colab runtime

In [ ]:
import shutil
import subprocess

if not TRAIN_DIR.exists() or not VAL_DIR.exists():
    usage = shutil.disk_usage('/content')
    print(f'พื้นที่ว่างใน Colab: {usage.free / 1024**3:.1f} GB')
    if usage.free < 12 * 1024**3:
        raise RuntimeError('พื้นที่ว่างน้อยกว่า 12 GB กรุณา Restart runtime แล้วลองใหม่')

    if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != ZIP_PATH.stat().st_size:
        print('กำลังคัดลอก ZIP จาก Drive มายัง local SSD ของ Colab...')
        shutil.copy2(ZIP_PATH, LOCAL_ZIP)

    patterns = [
        'TH_Characters/Train_Test-Base/labels.csv',
        'TH_Characters/Train_Test-Base/split_80_20/summary_aug.csv',
        'TH_Characters/Train_Test-Base/split_80_20/train_aug/*',
        'TH_Characters/Train_Test-Base/split_80_20/test/*',
    ]
    print('กำลังแตกเฉพาะ Train/Validation ที่ต้องใช้...')
    subprocess.run(
        ['unzip', '-q', str(LOCAL_ZIP), *patterns, '-d', str(EXTRACT_ROOT)],
        check=True,
    )
else:
    print('พบข้อมูลที่แตกไว้แล้ว ข้ามขั้นตอน unzip')

assert TRAIN_DIR.exists() and VAL_DIR.exists()
print('Train:', TRAIN_DIR)
print('Validation:', VAL_DIR)


## 3. Imports และการทำซ้ำผลการทดลอง

กำหนด seed เดียวกับทีม Data (`42`) และเปิด mixed precision บน GPU เพื่อลดเวลาและหน่วยความจำ

In [ ]:
import copy
import json
import math
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from torchvision.models import MobileNet_V3_Large_Weights

ImageFile.LOAD_TRUNCATED_IMAGES = True
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('ไม่พบ GPU กรุณาเลือก T4 GPU ใน Runtime settings')

torch.backends.cudnn.benchmark = True
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0))


## 4. สำรวจชุดข้อมูล

ชุดข้อมูลมี 80 คลาส เป็นพยัญชนะ สระ วรรณยุกต์ เครื่องหมาย และตัวเลขไทย การแบ่งข้อมูลเป็นแบบ group-aware เพื่อไม่ให้ font หรือผู้เขียนคนเดียวกันหลุดข้าม Train/Validation

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)
display(labels_df.head(10))
display(summary_df.head())
print(f'จำนวนคลาส: {len(labels_df)}')
print(f'ภาพ Train + Augmentation: {summary_df.train_aug.sum():,}')
print(f'ภาพ Validation: {summary_df.test.sum():,}')
print('จำนวนคลาสแยกตามประเภท:')
display(labels_df.groupby('category').size().rename('n_classes').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=summary_df, x='class', y='train_aug', ax=axes[0], color='#2A9D8F')
axes[0].set_title('Train images per class (with offline augmentation)')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)
sns.barplot(data=summary_df, x='class', y='test', ax=axes[1], color='#E76F51')
axes[1].set_title('Validation images per class')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
plt.tight_layout()
plt.show()


## 5. Data pipeline และ Letterbox

ภาพต้นฉบับมีขนาดประมาณ 3-80 px และสัดส่วนของตัวอักษรมีความหมาย จึงใช้ letterbox แทนการยืดภาพเป็นสี่เหลี่ยม สำหรับ Data Augmentation ใช้ภาพที่ทีม Data สร้างไว้ใน `train_aug` อยู่แล้ว จึงปิด online augmentation เป็นค่าเริ่มต้นเพื่อไม่ให้ดัดแปลงซ้ำมากเกินไป

In [ ]:
IMAGE_SIZE = 160
BATCH_SIZE = 128
NUM_WORKERS = 2
ONLINE_AUGMENTATION = False

class Letterbox:
    def __init__(self, size, fill=(255, 255, 255)):
        self.size = (size, size)
        self.fill = fill

    def __call__(self, image):
        image = image.convert('RGB')
        return ImageOps.pad(
            image, self.size, method=Image.Resampling.BILINEAR,
            color=self.fill, centering=(0.5, 0.5)
        )

normalize = transforms.Normalize(
    mean=MobileNet_V3_Large_Weights.DEFAULT.transforms().mean,
    std=MobileNet_V3_Large_Weights.DEFAULT.transforms().std,
)

train_ops = [Letterbox(IMAGE_SIZE)]
if ONLINE_AUGMENTATION:
    train_ops += [
        transforms.RandomAffine(degrees=5, translate=(0.04, 0.04), scale=(0.95, 1.05), fill=255),
    ]
train_ops += [transforms.ToTensor(), normalize]

train_transform = transforms.Compose(train_ops)
eval_transform = transforms.Compose([Letterbox(IMAGE_SIZE), transforms.ToTensor(), normalize])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)

assert train_dataset.classes == val_dataset.classes
assert len(train_dataset.classes) == 80
expected_codes = labels_df.tis620.astype(str).tolist()
assert train_dataset.classes == expected_codes, 'ลำดับคลาสไม่ตรงกับ labels.csv'

loader_kwargs = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

print('Train:', len(train_dataset), 'Validation:', len(val_dataset))
print('Classes:', len(train_dataset.classes))


In [ ]:
# แสดงตัวอย่างหลังผ่าน letterbox/normalization (ย้อน normalize เพื่อแสดงผล)
images, targets = next(iter(train_loader))
mean = torch.tensor(MobileNet_V3_Large_Weights.DEFAULT.transforms().mean).view(3, 1, 1)
std = torch.tensor(MobileNet_V3_Large_Weights.DEFAULT.transforms().std).view(3, 1, 1)

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for ax, image, target in zip(axes.flat, images[:18], targets[:18]):
    shown = (image.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
    row = labels_df.iloc[target.item()]
    ax.imshow(shown)
    ax.set_title(f"{row['char']} ({row['tis620']})")
    ax.axis('off')
plt.tight_layout()
plt.show()


## 6. MobileNetV3-Large และ Loss

ใช้ ImageNet pretrained weights แล้วเปลี่ยน classifier เป็น 80 คลาส Class weights คำนวณจาก Train เท่านั้น และ label smoothing ช่วยลดความมั่นใจเกินจริงในคลาสที่หน้าตาคล้ายกัน

In [ ]:
NUM_CLASSES = len(train_dataset.classes)
class_counts = np.bincount(train_dataset.targets, minlength=NUM_CLASSES)
class_weights = len(train_dataset) / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.10)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


## 7. Training utilities

เก็บโมเดลที่ Validation Accuracy สูงสุด ใช้ mixed precision และ early stopping โดยดู Validation Loss

In [ ]:
history = []
best_val_acc = -1.0
best_state = None

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_items = 0
    scaler = torch.amp.GradScaler('cuda', enabled=training)

    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
                logits = model(inputs)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == targets).sum().item()
        total_items += batch_size

    return total_loss / total_items, total_correct / total_items

def train_stage(stage_name, epochs, optimizer, patience=3):
    global best_val_acc, best_state
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.3, patience=1, min_lr=1e-7
    )
    stage_best_loss = math.inf
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        started = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        scheduler.step(val_loss)
        elapsed = time.time() - started
        lrs = [group['lr'] for group in optimizer.param_groups]

        row = {
            'stage': stage_name, 'epoch': epoch,
            'train_loss': train_loss, 'train_acc': train_acc,
            'val_loss': val_loss, 'val_acc': val_acc,
            'lr': max(lrs), 'minutes': elapsed / 60,
        }
        history.append(row)
        print(
            f"[{stage_name}] {epoch:02d}/{epochs} | "
            f"train {train_loss:.4f}/{train_acc:.2%} | "
            f"val {val_loss:.4f}/{val_acc:.2%} | {elapsed/60:.1f} min"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save({
                'model_state_dict': best_state,
                'class_to_idx': train_dataset.class_to_idx,
                'image_size': IMAGE_SIZE,
                'best_val_acc': best_val_acc,
            }, OUTPUT_DIR / 'mobilenetv3_large_best.pt')

        if val_loss < stage_best_loss - 1e-4:
            stage_best_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping: validation loss ไม่ดีขึ้น {patience} epochs')
                break

    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)


## 8. Stage 1: ฝึกเฉพาะ Classifier

Freeze feature extractor ก่อน เพื่อให้หัว classifier ใหม่ปรับตัวโดยไม่ทำลาย pretrained features

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

optimizer_stage1 = torch.optim.AdamW(
    model.classifier.parameters(), lr=1e-3, weight_decay=1e-4
)
train_stage('classifier', epochs=3, optimizer=optimizer_stage1, patience=2)


## 9. Stage 2: Fine-tune Backbone

Unfreeze ทั้งโมเดล แต่ให้ backbone ใช้ learning rate ต่ำกว่า classifier 10 เท่า เพื่อปรับ feature ให้เข้ากับอักษรไทยอย่างค่อยเป็นค่อยไป

In [ ]:
for param in model.features.parameters():
    param.requires_grad = True

optimizer_stage2 = torch.optim.AdamW([
    {'params': model.features.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)
train_stage('fine_tune', epochs=12, optimizer=optimizer_stage2, patience=4)

model.load_state_dict(best_state)
print(f'Best validation accuracy: {best_val_acc:.2%}')


## 10. Learning curves

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(1, len(history_df) + 1)
axes[0].plot(x, history_df.train_loss, marker='o', label='Train')
axes[0].plot(x, history_df.val_loss, marker='o', label='Validation')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy loss')
axes[0].legend()
axes[1].plot(x, history_df.train_acc * 100, marker='o', label='Train')
axes[1].plot(x, history_df.val_acc * 100, marker='o', label='Validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves.png', dpi=180, bbox_inches='tight')
plt.show()


## 11. ประเมิน Validation และวิเคราะห์คลาสที่สับสน

รายงาน Accuracy, precision, recall, F1-score, confusion matrix และคู่คลาสที่ทำนายสับสนบ่อยที่สุด

In [ ]:
@torch.inference_mode()
def predict_loader(model, loader):
    model.eval()
    all_targets, all_preds, all_probs = [], [], []
    for inputs, targets in loader:
        inputs = inputs.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(inputs)
        probs = logits.softmax(1).cpu()
        all_targets.extend(targets.numpy())
        all_preds.extend(probs.argmax(1).numpy())
        all_probs.extend(probs.max(1).values.numpy())
    return np.array(all_targets), np.array(all_preds), np.array(all_probs)

y_true, y_pred, confidence = predict_loader(model, val_loader)
val_accuracy = accuracy_score(y_true, y_pred)
target_names = [f"{row.char} ({row.tis620})" for row in labels_df.itertuples()]
report = classification_report(
    y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0
)
report_df = pd.DataFrame(report).T
report_df.to_csv(OUTPUT_DIR / 'classification_report.csv')
print(f'Validation accuracy: {val_accuracy:.2%}')
display(report_df.tail(5))


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
plt.figure(figsize=(24, 20))
sns.heatmap(cm, cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title(f'MobileNetV3-Large Confusion Matrix | Accuracy {val_accuracy:.2%}')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()

cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)
pairs = []
for flat_idx in np.argsort(cm_errors.ravel())[::-1][:20]:
    true_idx, pred_idx = np.unravel_index(flat_idx, cm_errors.shape)
    if cm_errors[true_idx, pred_idx] == 0:
        break
    pairs.append({
        'true': target_names[true_idx],
        'predicted': target_names[pred_idx],
        'count': int(cm_errors[true_idx, pred_idx]),
    })
top_confusions_df = pd.DataFrame(pairs)
top_confusions_df.to_csv(OUTPUT_DIR / 'top_confusions.csv', index=False)
display(top_confusions_df)


In [ ]:
metrics = {
    'model': 'MobileNetV3-Large',
    'pretrained': 'ImageNet',
    'seed': SEED,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'train_images': len(train_dataset),
    'validation_images': len(val_dataset),
    'num_classes': NUM_CLASSES,
    'validation_accuracy': float(val_accuracy),
    'macro_f1': float(report['macro avg']['f1-score']),
    'weighted_f1': float(report['weighted avg']['f1-score']),
    'techniques': [
        'transfer learning', 'offline data augmentation', 'letterbox resize',
        'class-weighted loss', 'label smoothing', 'mixed precision',
        'two-stage fine-tuning', 'early stopping'
    ],
}
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)
display(pd.Series(metrics))


## 12. ทำนายชุด Test 5% ของอาจารย์

นำภาพ Test ของอาจารย์ใส่ไว้ใต้ `HIDDEN_TEST_DIR` จะวางเป็นโฟลเดอร์เดียวหรือมีโฟลเดอร์ย่อยก็ได้ Cell นี้ไม่ใช้ผล Test เพื่อปรับโมเดล และจะสร้าง `submission.csv` สำหรับส่งผลทำนาย

In [ ]:
MOCK_TEST_ZIP = Path('/content/drive/MyDrive/dl/TH_Characters_Test_5pct.zip')
HIDDEN_TEST_DIR = Path('/content/TH_Characters_Test_5pct')
if not HIDDEN_TEST_DIR.exists() and MOCK_TEST_ZIP.exists():
    print('กำลังแตก Mock Test จาก Google Drive...')
    shutil.unpack_archive(MOCK_TEST_ZIP, '/content')

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

class UnlabeledImages(Dataset):
    def __init__(self, root, transform):
        self.root = Path(root)
        self.files = sorted(
            path for path in self.root.rglob('*')
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        )
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        path = self.files[index]
        with Image.open(path) as image:
            tensor = self.transform(image)
        return tensor, str(path.relative_to(self.root))

if not HIDDEN_TEST_DIR.exists():
    print(f'ยังไม่พบชุด Test: {HIDDEN_TEST_DIR}')
    print('แก้ HIDDEN_TEST_DIR แล้วรัน cell นี้อีกครั้งเมื่อได้รับชุด Test จากอาจารย์')
else:
    hidden_dataset = UnlabeledImages(HIDDEN_TEST_DIR, eval_transform)
    hidden_loader = DataLoader(hidden_dataset, shuffle=False, **loader_kwargs)
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, filenames in hidden_loader:
            inputs = inputs.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                probs = model(inputs).softmax(1).cpu()
            pred_indices = probs.argmax(1).numpy()
            confidences = probs.max(1).values.numpy()
            for filename, pred_idx, conf in zip(filenames, pred_indices, confidences):
                label = labels_df.iloc[int(pred_idx)]
                rows.append({
                    'filename': filename,
                    'class_id': int(pred_idx),
                    'tis620': int(label.tis620),
                    'char': label.char,
                    'name_en': label.name_en,
                    'confidence': float(conf),
                })
    submission_df = pd.DataFrame(rows)
    submission_df.to_csv(OUTPUT_DIR / 'submission.csv', index=False, encoding='utf-8-sig')
    print(f'ทำนายแล้ว {len(submission_df):,} ภาพ')
    display(submission_df.head())


## 13. ตรวจคะแนน Mock Test (ถ้ามีเฉลย)

Cell นี้ใช้เฉพาะ `ground_truth.csv` ที่มากับชุด mock ของเรา ห้ามใช้กับ Test จริงของอาจารย์ และห้ามนำคะแนนนี้ไปรายงานเป็น unseen test accuracy

In [ ]:
ground_truth_path = HIDDEN_TEST_DIR / 'ground_truth.csv'

if 'submission_df' not in globals():
    print('กรุณารัน Cell ทำนายชุด Test ก่อน')
elif not ground_truth_path.exists():
    print('ไม่พบ ground_truth.csv: ข้ามการตรวจคะแนนสำหรับชุด Test จริง')
else:
    truth_df = pd.read_csv(ground_truth_path)
    predictions_df = submission_df.copy()
    predictions_df['filename'] = predictions_df.filename.map(lambda value: Path(value).name)
    scored_df = truth_df.merge(
        predictions_df[['filename', 'tis620', 'confidence']],
        on='filename', how='inner', validate='one_to_one'
    )
    scored_df['correct'] = scored_df['class'].astype(int) == scored_df['tis620'].astype(int)
    mock_accuracy = scored_df.correct.mean()
    print(f'Mock test images matched: {len(scored_df):,}/{len(truth_df):,}')
    print(f'Mock test accuracy: {mock_accuracy:.2%}')
    scored_df.to_csv(OUTPUT_DIR / 'mock_test_scored.csv', index=False, encoding='utf-8-sig')
    display(
        scored_df.groupby(['class', 'char']).correct.mean()
        .sort_values().rename('accuracy').head(15).to_frame()
    )


## ประเด็นสำหรับนำเสนอ

- **Dataset:** 80 คลาส รวมพยัญชนะ สระ วรรณยุกต์ เครื่องหมาย และตัวเลขไทย
- **ความท้าทาย:** คลาสไม่สมดุล, ภาพเล็ก, font/ลายมือหลากหลาย และตัวอักษรบางคู่คล้ายกันมาก
- **Data split:** 80/20 แบบ group-aware ด้วย seed 42 ป้องกัน font และผู้เขียนรั่วข้ามชุด
- **Transfer Learning:** MobileNetV3-Large pretrained จาก ImageNet แล้วเปลี่ยน classifier เป็น 80 outputs
- **Data Augmentation:** ทีม Data สร้าง 10 รูปแบบจาก Train เท่านั้น และรวมไว้ใน `train_aug`
- **เทคนิคเพิ่มเติม:** letterbox, class weights, label smoothing, mixed precision, early stopping และ fine-tuning สองช่วง
- **ผลลัพธ์:** ใช้ learning curves, Validation Accuracy/F1, confusion matrix และ top confused pairs จากโฟลเดอร์ outputs
- **การเปรียบเทียบ:** ทั้งสามโมเดลต้องใช้ split, seed และนิยาม metric เดียวกัน